# Week 1 Homework – Reinforcement Learning for Data Science Club

**From Prediction to Decision Making: Markov Decision Processes**

**Due**: Before Week 4 live session  
**Goal**: Build intuition for MDPs by implementing core concepts on two gridworld environments.  
**Estimated time**: 30-60 mins 

We will work with two environments:
- **FrozenLake-v1** (non-slippery): 4×4 grid to get familiar
- **A custom 4×4 Gridworld**: Slightly different rewards/transitions for exact solving practice

**Submit**: Run all cells → download .ipynb or export to PDF → upload to [Google Drive / Discord / club platform]

---

## Instructions

Fill in all `# TODO` sections.  
Do **not** change provided function signatures or test cells unless told otherwise.  
We use simple `assert` checks — they must pass for full credit.

Good luck — this notebook directly builds on the Week 1 lecture!

## 0. Setup & Dependencies

In [ ]:
# !pip install gymnasium matplotlib numpy  # uncomment if needed

import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from typing import Callable

%matplotlib inline

## 1. FrozenLake Environment (Non-slippery)

In [ ]:
env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="ansi")

print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
print("\nInitial state:")
print(env.reset())
print(env.render())

# Understanding env.P (transition probabilities):
# env.P[state][action] returns a list of possible transitions
# Each transition is a tuple: (probability, next_state, reward, done)
# Example: env.P[0][0] might return [(1.0, 0, 0.0, False)] 
#          meaning: from state 0, action 0 leads to state 0 with prob 1.0, reward 0, not done
print("\nExample transition from state 0, action 0:")
print(env.P[0][0])

### Part 1.1 – Random Policy Rollout

In [ ]:
def random_policy(state: int) -> int:
    """Select action uniformly at random."""
    # TODO: return a random action between 0 and env.action_space.n - 1
    # HINT: Use np.random.randint(low, high) where high is exclusive
    #       The action space has env.action_space.n possible actions (0 to n-1)
    pass


# Test your random policy (should look random)
print("Sample actions:", [random_policy(0) for _ in range(10)])

In [ ]:
def evaluate_policy(policy: Callable, env, n_episodes=200, max_steps=100, seed=42):
    env = gym.make("FrozenLake-v1", is_slippery=False)  # fresh env
    successes = 0
    for ep in range(n_episodes):
        state, _ = env.reset(seed=seed + ep)
        done = False
        steps = 0
        while not done and steps < max_steps:
            action = policy(state)
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            steps += 1
        if reward > 0.5:  # reached goal
            successes += 1
    return successes / n_episodes


random_success_rate = evaluate_policy(random_policy, env)
print(f"Random policy success rate: {random_success_rate:.3f}")

assert 0.00 < random_success_rate < 0.20, "Random policy should succeed rarely"

### Part 1.2 – Policy Evaluation (compute V^π)

**Key concept**: The Bellman equation for policy evaluation:
$$V^\pi(s) = \sum_a \pi(a|s) \sum_{s',r} p(s',r|s,a) [r + \gamma V^\pi(s')]$$

For a random (uniform) policy: $\pi(a|s) = 1/n_{actions}$ for all actions.

In [ ]:
def policy_evaluation(policy: Callable, env, gamma=0.99, theta=1e-6):
    """
    Iteratively evaluate the value function V for a given policy.
    Returns: V (numpy array of shape [n_states])
    """
    n_states = env.observation_space.n
    V = np.zeros(n_states)

    while True:
        delta = 0
        for s in range(n_states):
            v_old = V[s]
            # TODO: compute new V[s] = expected [r + gamma * V[s']]
            # 
            # HINTS:
            # 1. Check if state s is terminal: if len(env.P[s]) == 0, set V[s] = 0.0 and continue
            # 2. For non-terminal states, initialize v_new = 0.0
            # 3. Loop over all actions a (0 to env.action_space.n - 1)
            #    - For random policy, action probability = 1.0 / env.action_space.n
            # 4. For each action, get transitions: env.P[s][a] 
            #    - This is a list of tuples: [(prob, next_state, reward, done), ...]
            # 5. Loop over each transition tuple (prob, next_state, reward, done):
            #    - If done is True: add action_prob * prob * reward to v_new
            #    - If done is False: add action_prob * prob * (reward + gamma * V[next_state]) to v_new
            # 6. Set V[s] = v_new
            # ...
            V[s] = ...  # your computation here
            delta = max(delta, abs(v_old - V[s]))
        if delta < theta:
            break

    return V


# Evaluate random policy
V_random = policy_evaluation(random_policy, env)
print("Value function for random policy (4×4):")
print(V_random.reshape(4, 4).round(3))

In [ ]:
# Visualize
plt.figure(figsize=(6,5))
plt.imshow(V_random.reshape(4,4), cmap='viridis')
plt.colorbar(label='Value')
plt.title("V^π (random policy) on FrozenLake")
plt.xticks(range(4)); plt.yticks(range(4))
plt.show()

### Part 1.3 – Greedy Policy from V^π

**Key concept**: Q-value is the expected return from taking action a in state s:
$$Q(s,a) = \sum_{s',r} p(s',r|s,a) [r + \gamma V(s')]$$

The greedy policy selects: $\pi(s) = \arg\max_a Q(s,a)$

In [ ]:
def get_greedy_policy(V: np.ndarray, env, gamma=0.99):
    """Extract deterministic greedy policy from value function."""
    n_states = len(V)
    policy = np.zeros(n_states, dtype=int)

    for s in range(n_states):
        q_values = []
        for a in range(env.action_space.n):
            # TODO: compute Q(s,a) = expected [r + gamma * V[s']]
            # 
            # HINTS:
            # 1. Initialize q = 0.0
            # 2. Get transitions for this state-action: env.P[s][a]
            #    - This is a list of tuples: [(prob, next_state, reward, done), ...]
            # 3. Loop over each transition tuple (prob, next_state, reward, done):
            #    - If done is True: add prob * reward to q
            #    - If done is False: add prob * (reward + gamma * V[next_state]) to q
            # 4. Append q to q_values (already done below)
            q = 0.0
            # ...
            q_values.append(q)
        policy[s] = np.argmax(q_values)

    return policy


policy_greedy_random = get_greedy_policy(V_random, env)
print("Greedy policy from random V (action per state):")
print(policy_greedy_random.reshape(4,4))

In [ ]:
greedy_success_rate = evaluate_policy(
    lambda s: policy_greedy_random[s], env
)
print(f"Greedy-from-random success rate: {greedy_success_rate:.3f}")

assert greedy_success_rate > random_success_rate, \
    "Greedy policy should be better than random"

## 2. Value Iteration – Solve FrozenLake Exactly

**Key concept**: Value iteration uses the Bellman optimality equation:
$$V^*(s) = \max_a \sum_{s',r} p(s',r|s,a) [r + \gamma V^*(s')]$$

This is similar to policy evaluation, but we take the **maximum** over actions instead of averaging.

In [ ]:
def value_iteration(env, gamma=0.99, theta=1e-6):
    n_states = env.observation_space.n
    V = np.zeros(n_states)

    while True:
        delta = 0
        for s in range(n_states):
            v_old = V[s]
            # TODO: V[s] = max_a expected [r + gamma V[s']]
            # 
            # HINTS:
            # 1. Check if state s is terminal: if len(env.P[s]) == 0, set V[s] = 0.0 and continue
            # 2. For non-terminal states, initialize max_q = float('-inf') or a very small number
            # 3. Loop over all actions a (0 to env.action_space.n - 1)
            # 4. For each action, compute Q(s,a):
            #    - Initialize q = 0.0
            #    - Get transitions: env.P[s][a] (list of tuples)
            #    - Loop over transitions (prob, next_state, reward, done):
            #      * If done: q += prob * reward
            #      * If not done: q += prob * (reward + gamma * V[next_state])
            #    - Update max_q = max(max_q, q)
            # 5. Set V[s] = max_q
            # ...
            V[s] = ...  # your code
            delta = max(delta, abs(v_old - V[s]))
        if delta < theta:
            break

    return V


V_opt = value_iteration(env)
print("Optimal V* (4×4):")
print(V_opt.reshape(4,4).round(3))

In [ ]:
# Extract optimal policy
policy_opt = get_greedy_policy(V_opt, env)
print("Optimal policy (0=←, 1=↓, 2=→, 3=↑):")
print(policy_opt.reshape(4,4))

In [ ]:
# Should be close to 1.0
opt_success_rate = evaluate_policy(
    lambda s: policy_opt[s], env, n_episodes=50
)
print(f"Optimal policy success rate: {opt_success_rate:.3f}")

assert V_opt[15] > 0.999, "Goal state should have value ≈ 1.0"
assert opt_success_rate > 0.95, "Optimal policy should almost always succeed"

## 3. Reflection Questions (short answer)

**Question 1** (2–4 sentences):  
Why does the optimal value function give non-zero values to some frozen tiles even though they give 0 immediate reward?

*Hint: Think about what the value function represents. Is it just the immediate reward, or does it consider future rewards? What happens when you're on a frozen tile that's on the path to the goal?*

**Your answer here**:



**Question 2** (2–4 sentences):  
Compare the success rates: random vs. greedy-from-random vs. optimal.  
What does this tell you about the importance of **planning** (using value functions) vs. pure exploration?

*Hint: Consider what each policy does differently. Random policy explores without guidance. Greedy-from-random uses a value function (even if suboptimal). Optimal uses the best value function. What does this suggest about the value of computing value functions?*

**Your answer here**:



**Question 3** (optional bonus):  
If we made the lake slippery again (`is_slippery=True`), how do you think the optimal policy and values would change? Why?

*Hint: Slippery means stochastic transitions - actions don't always work as intended. How does uncertainty affect the expected value? Would you expect higher or lower values? Would the optimal actions change?*

**Your answer here** (optional):



---

## Congratulations!

You have now:
- Implemented policy evaluation
- Extracted greedy policies
- Solved an MDP exactly with value iteration

Next week (Tabular RL) we drop the known transition model `P` and learn by interacting — see you then!